## NB3. ROF Global Daily flow anaysis at HB14 gauges (only gauges on river network) <a id='top'></a>

Use 

    1. reach-HB14 gauge link ascii
    2. HB14 flow site shapefile
    3. HB14 discharge netCDF
    4. daily flow netCD (history file at only gauge) <case>.h.yyyy_daily_gauge.nc

[1. Setupt](#setup)

[2. Loading data and preprocess](#load_data)

- daily history files (directory from CESM or postprocessed) from archive. 

- Reference data is daily discharge estimates at xxxx big river mouths from Beck et al. 2014 data (HB14)

[3. Compute daily metrics at gauges](#Compute_metrics)

[4. Plot](#plot)

In [ ]:
%matplotlib inline

import os, sys
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FormatStrFormatter
from sklearn.linear_model import LinearRegression
from scipy import stats
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask_jobqueue import PBSCluster
from dask.distributed import Client

import scripts.metrics as metrics
import scripts.colors as colors
from scripts.utility import load_yaml
from scripts.utility import reorder_index
from scripts.utility import AutoVivification

print("\nThe Python version: %s.%s.%s" % sys.version_info[:3])

-------------------------
## 1. Analysis setup <a id='setup'></a>

**Please provide CESM case names and ROF grid name**

[go back to top](#top)

In [ ]:
# CESM case names and their runoff grid
plot_name = "test"

cases = {
        #'f09_f09_rHDMA':'rHDMA',
        #'f09_f09_rHDMAlk':'rHDMAlk',
        'f09_f09_rHDMAlk_h06':'rHDMAlk_h06',
        #'f09_f09_rMERIT':'rMERIT',
        'f09_f09_mg17_mosart':'f09_f09_mosart',
        }

error_metric = 'mae'

parallel = False

figureSave = False

-------------------------
load config files and some parameters 

In [ ]:
setup = load_yaml('./setup/setup.yaml')

main_dir       = setup['archive_dir']     # CESM archive directory
domain_dir     = setup['ancillary_dir']   # ancillary directory including such as ROF domain 
geospatial_dir = setup['ancillary_dir']   # including shapefiles etc
ancillary_dir  = setup['ancillary_dir']
ref_flow_dir   = setup['ref_flow_dir']    # including observed or reference flow data 
syr            = setup['syr']             # analysis start year
eyr            = setup['eyr']             # analysis end year
case_meta      = setup["case_meta"]  # RO grid meta
reach_gpkg     = setup['reach_gpkg']      # reach geopackage meta

network = setup['river_network']

time_period = slice(f'{syr}-01-01',f'{eyr}-12-31') # analysis time period
nyrs = eyr-syr+1  # number of years
nmons = nyrs*12   # number of months

-----
### dasks (optional)

In [ ]:
if parallel:
    cluster = PBSCluster(queue='casper', memory='10GB', processes=1)
    cluster.scale(jobs=5)
    client = Client(cluster)
    client

-------------------------
## 2. Loading data and preprocess <a id='load_data'></a>

[go back to top](#top)

### 2.1. Read simulated discharge data (at only gauge points)

In [ ]:
%%time
reachID    = {}
daily_data = {}
for case, grid_name in cases.items():
    in_dire = os.path.join(main_dir, case, 'rof/hist')
    model  = case_meta[grid_name]['model']
    #daily
    daily_data[case] = xr.open_mfdataset(f'{in_dire}/{case}.{model}.*.HB14.nc', data_vars='minimal').sel(time=time_period).load()
        
    print(f"reading {case}...")

In [ ]:
f'{in_dire}/{case}.{model}.*.HB14.nc'

### 2.2 HB14 discharge data
- ds_q_obs_daily (xr datasets)

**open_dataset with decode_times=True outputs lazy loading, so need load method to bring the dataset into memory**

In [ ]:
%%time
ds_q_obs_daily = xr.open_dataset('%s/HB14/HB14_daily_discharge.nc'%(ref_flow_dir), decode_times=True).sel(time=time_period).load()
ds_q_obs_daily = ds_q_obs_daily.sel(time=~((ds_q_obs_daily.time.dt.month == 2) & (ds_q_obs_daily.time.dt.day == 29)))
ds_q_obs_daily['time'] = xr.cftime_range(start=time_period.start, end=time_period.stop, freq="D", calendar="noleap")

### 2.3 HB14 flow site shapefile
- gauge_shp (dataframe)

In [ ]:
%%time
gauge_shp = gpd.read_file(os.path.join(ref_flow_dir, 'HB14','geospatial','HB14.shp'))
gauge_shp = gauge_shp[gauge_shp['id']!=9999999]

### 2.4 Read river network netCDF
- riv_network (df)

In [ ]:
reach_properties =['seg_id','upsArea']
riv_network = {}
for case, grid_name in cases.items():
    
    network_name = case_meta[grid_name]['network']
    network_file = os.path.join(ancillary_dir, network[network_name]['file_name'])  # geopackage name
    
    ds_rn = xr.open_dataset(network_file)[reach_properties].set_index(seg='seg_id')
    riv_network[case] = ds_rn.to_dataframe()

    print(f"reading {network_file}...")

### 2.5. reach-HB14 gauge link csv
- gauge_reach_lnk (dataframe)
- ds_gauge (dataset)

In [ ]:
gauge_reach_lnk = {}
ds_gauge ={}   # used for reorder
for case, grid_name in cases.items():
    gauge_reach_lnk[case] = pd.read_csv('%s/HB14/HB14.%s.asc'%(ref_flow_dir, case_meta[grid_name]['network']))
    gauge_reach_lnk[case].rename(columns={"route_id": "reachID"}, inplace=True)

    # remove headwater gauges
    df_tmp = pd.merge(gauge_reach_lnk[case], riv_network[case], left_on='reachID', right_index=True)
    gauge_reach_lnk[case] = df_tmp.loc[df_tmp['upsArea'] > 0.0] 
    print('%s: %d -> %d after headwater removed'%(case, len(df_tmp), len(gauge_reach_lnk[case])))

    ds_gauge[case] = gauge_reach_lnk[case][['gauge_id','reachID']].set_index('gauge_id').to_xarray()
    ds_gauge[case] = ds_gauge[case].rename({'gauge_id':'gauge'},)

**If there are more than 1 cases, find common gauges among all the networks**

- gauge_reach_lnk_common
- ds_gauge_common (dataset)

In [ ]:
case_gauge_list = [gauge_reach_lnk[case]['gauge_id'].values for case in cases.keys()]
common_gauge = list(set(case_gauge_list[0]).intersection(*case_gauge_list))

gauge_reach_lnk_common = {}
ds_gauge_common ={}

for ix, case in enumerate(cases.keys()):
    gauge_reach_lnk_common[case] = gauge_reach_lnk[case][gauge_reach_lnk[case]["gauge_id"].isin(common_gauge)]
    ds_gauge_common[case] = ds_gauge[case].where(ds_gauge[case].gauge.isin(common_gauge), drop=True)
    if ix == 0:
        gauge_id_network_common = gauge_reach_lnk_common[case]['gauge_id'].values
    else:
        if not ((gauge_id_network_common - gauge_reach_lnk_common[case]['gauge_id'].values)==0).all():
            print('common gauge id inconsistency')

### 2.6 Subset simulated flow data and observed flow data at common gauges

In [ ]:
# Extract only at common gauge
reach_id_network_common = {}
daily_data_common = {}
for case, grid_name in cases.items():
    daily_data_common[case] = daily_data[case].where(daily_data[case]['reachID'].isin(gauge_reach_lnk_common[case]['reachID']), drop=True)
    reach_id_network_common[case] = daily_data_common[case].reachID.values

In [ ]:
%%time
# observed discharge subset
daily_q_obs_daily_network = {}
for case, grid_name in cases.items():
    daily_q_obs_daily_network[case] = ds_q_obs_daily.sel(gauge=ds_gauge[case].gauge)
    
ds_q_obs_daily_common = ds_q_obs_daily.sel(gauge=gauge_id_network_common)

### 2.7 Reorder sim and obs to match up with ds_gage


For gauge resolving individual networks
- daily_data
- ds_q_obs_daily

In [ ]:
%%time
for case, grid_name in cases.items():

    remap_order = reorder_index(daily_data[case]['reachID'].values, ds_gauge[case]['reachID'].values)
    # Reorder pio according to the orginal
    daily_data[case] = daily_data[case].isel(dict(seg=remap_order))

    remap_order = reorder_index(daily_q_obs_daily_network[case]['gauge'].values, ds_gauge[case]['gauge'].values)
    # Reorder pio according to the orginal
    daily_q_obs_daily_network[case] = daily_q_obs_daily_network[case].isel(dict(gauge=remap_order))

For common gauge resolving all the networks
- daily_data_common
- ds_q_obs_daily_common

In [ ]:
%%time
for case, grid_name in cases.items():

    remap_order = reorder_index(daily_data_common[case]['reachID'].values, ds_gauge_common[case]['reachID'].values)
    # Reorder pio according to the orginal
    daily_data_common[case] = daily_data_common[case].isel(dict(seg=remap_order))

remap_order = reorder_index(ds_q_obs_daily_common['gauge'].values, ds_gauge_common[case]['gauge'].values)
# Reorder pio according to the orginal
ds_q_obs_daily_common = ds_q_obs_daily_common.isel(dict(gauge=remap_order))

----
some checks. should get all **True**

In [ ]:
((daily_data_common['f09_f09_rHDMA']['reachID'].values - ds_gauge_common['f09_f09_rHDMA']['reachID'].values)==0).all()

In [ ]:
((ds_q_obs_daily_common['gauge']- ds_gauge_common['f09_f09_mg17_mosart']['gauge'])==0).values.all()

In [ ]:
((ds_gauge_common['f09_f09_rHDMA']['gauge'] - ds_gauge_common['f09_f09_mg17_mosart']['gauge'])==0).values.all()

-------------------------
## 3. Compute daily metrics (correlation, MEA, pbias) at gauges <a id='Compute_metrics'></a>

[go back to top](#top)

**use all the guages available for each network**

In [ ]:
%%time

daily_metrics = AutoVivification()

for case, grid_name in cases.items():

    q_name=case_meta[grid_name]['flow_name']
    
    ngauge = len(ds_gauge[case].gauge)
    
    corr_array  = np.full(ngauge, np.nan, dtype=float)
    mae_array   = np.full(ngauge, np.nan, dtype=float)
    pbias_array = np.full(ngauge, np.nan, dtype=float)
    
    for ix in np.arange(ngauge):
        mae_value   = metrics.mae(daily_data[case][q_name][:,ix].values, daily_q_obs_daily_network[case]['daily_discharge'][:,ix].values)
        corr_value  = metrics.corr(daily_data[case][q_name][:,ix].values, daily_q_obs_daily_network[case]['daily_discharge'][:,ix].values)
        pbias_value = metrics.pbias(daily_data[case][q_name][:,ix].values, daily_q_obs_daily_network[case]['daily_discharge'][:,ix].values)
        
        if np.isinf(mae_value):
            mae_value = np.nan
        if np.isinf(corr_value):
            corr_value = np.nan
        if np.isinf(pbias_value):
            pbias_value = np.nan
            
        corr_array[ix]  = corr_value
        mae_array[ix]   = mae_value
        pbias_array[ix] = pbias_value
        
    daily_metrics[case]['corr']  = corr_array
    daily_metrics[case]['mae']   = mae_array
    daily_metrics[case]['pbias'] = pbias_array

**Only at common gauges across different networks**

In [ ]:
%%time

daily_metrics_common = AutoVivification()
ngauge = len(gauge_id_network_common)

for case, grid_name in cases.items():

    q_name=case_meta[grid_name]['flow_name']
    
    corr_array = np.full(ngauge, np.nan, dtype=float)
    mae_array  = np.full(ngauge, np.nan, dtype=float)
    pbias_array = np.full(ngauge, np.nan, dtype=float)
    
    for ix in np.arange(ngauge):
        mae_value   = metrics.mae  (daily_data_common[case][q_name][:,ix].values, ds_q_obs_daily_common['daily_discharge'][:,ix].values)
        corr_value  = metrics.corr (daily_data_common[case][q_name][:,ix].values, ds_q_obs_daily_common['daily_discharge'][:,ix].values)
        pbias_value = metrics.pbias(daily_data_common[case][q_name][:,ix].values, ds_q_obs_daily_common['daily_discharge'][:,ix].values)
        
        if np.isinf(mae_value):
            mae_value = np.nan
        if np.isinf(corr_value):
            corr_value = np.nan
        if np.isinf(pbias_value):
            pbias_value = np.nan
            
        corr_array[ix] = corr_value
        mae_array[ix] = mae_value
        pbias_array[ix] = pbias_value
        
    daily_metrics_common[case]['corr']  = corr_array
    daily_metrics_common[case]['mae']   = mae_array
    daily_metrics_common[case]['pbias'] = pbias_array

-------------------------
## 4. plots <a id='plot'></a>

[go back to top](#top)

### 4.1 Spatial map of daily correlation

In [ ]:
# map
stat = 'corr'

cbar_kwrgs = {
              'pbias': {"shrink":0.9, "pad":0.02, "orientation": "horizontal", 'extend':'both'},
              'corr':  {"shrink":0.9, "pad":0.02, "orientation": "horizontal", 'extend':'min'},
              'rmse':  {"shrink":0.9, "pad":0.02, "orientation": "horizontal", 'extend':'max'},
              'mae':   {"shrink":0.9, "pad":0.02, "orientation": "horizontal", 'extend':'max'},
              'diff':  {"shrink":0.9, "pad":0.02, "orientation": "horizontal", 'extend':'both'},
              }

meta = {
    "pbias": {"name": "%bias", "vmin": -100, "vmax": 100, "cm": colors.cmap11},
    "corr": {"name": "correlation", "vmin": 0.2, "vmax": 1, "cm": colors.cmap12},
    "rmse": {"name": "RMSE", "vmin": 0, "vmax": 500, "cm": mpl.cm.turbo},
    "mae": {"name": "Mean Absolute Error", "vmin":0,"vmax":2,"cm":mpl.cm.turbo},
    "mae_diff": {"name": "MAE difference", "vmin":-1, "vmax":1, "cm":colors.cmap_diff},
    "corr_diff": {"name": "Corr difference", "vmin":-0.4, "vmax":0.4, "cm":colors.cmap_diff},
}

for case in cases.keys():
    if case=='f09_f09_mg17_mosart':
        ms=1.0
    else:
        ms=0.1
    
    fig = plt.figure(figsize=(7.5,4.0), dpi=100)
    ax = fig.add_subplot(1,1,1, projection=ccrs.PlateCarree())  
        
    ax.set_global()
    ax.coastlines(linewidth=0.5)
    
    pval = daily_metrics[case][stat]
    sc1 = ax.scatter(daily_q_obs_daily_network[case].gauge_lon, daily_q_obs_daily_network[case].gauge_lat, 
                     s=ms, c=pval, marker='o', cmap=meta[error_metric]['cm'], vmin=meta[error_metric]['vmin'], vmax=meta[error_metric]['vmax'], 
    )
    ax.set_extent([-180, 180, -60, 85])
    ax.set_title(f'{case} - {stat}')
    
    points = ax.collections[0]
    plt.colorbar(points, ax=ax, **cbar_kwrgs[error_metric]);
    
    plt.tight_layout()
    if figureSave:
        plt.savefig(f"NB3_Fig1_daily_{stat}_{case}_map.png", dpi=200)

In [ ]:
# map
base   = 'f09_f09_mg17_mosart'
target = 'f09_f09_rHDMAlk_mg17_dfw'
stat = 'corr'

if base=='f09_f09_mg17_mosart':
    ms=1.0
else:
    ms=0.2

fig = plt.figure(figsize=(7.5,4.0), dpi=100)
ax = fig.add_subplot(1,1,1, projection=ccrs.PlateCarree())  
    
ax.set_global()
ax.coastlines(linewidth=0.5)

pval = daily_metrics_common[target][stat] - daily_metrics_common[base][stat]
sc1 = ax.scatter(ds_q_obs_daily_common.gauge_lon, ds_q_obs_daily_common.gauge_lat, 
                 s=ms, c=pval, marker='o', cmap=meta[f'{stat}_diff']['cm'], vmin=meta[f'{stat}_diff']['vmin'], vmax=meta[f'{stat}_diff']['vmax'], 
)
ax.set_extent([-180, 180, -60, 85])
ax.set_title(f'{stat}: {target}-{base}')

points = ax.collections[0]
plt.colorbar(points, ax=ax, **cbar_kwrgs['diff']);

plt.tight_layout()
#plt.subplots_adjust(bottom=0.25, right=0.9, top=0.9)
plt.savefig(f"NB4_Fig1_daily_{stat}_{target}_vs_{base}_map.png", dpi=200)

### 4.2 Summary plots for daily metrics
- boxplots
- cdf

In [ ]:
# boxplots

stat = 'pbias'

column_stat = []
ngauge = len(gauge_id_network_common)
stat_array  = np.full((ngauge,len(cases)), np.nan, dtype='float')
for ix, (case, grid_name) in enumerate(cases.items()):
    column_stat.append(f"{stat}_{grid_name}")
    stat_array[:,ix] = daily_metrics_common[case][stat]
df = pd.DataFrame(stat_array, columns=column_stat)

boxprops    = {'linestyle':'-', 'linewidth':1.5, 'color':'blue'}
medianprops = {'linestyle':'-', 'linewidth':1.5, 'color':'red'}

fig, ax = plt.subplots(1, figsize=(6.5,4))
df.boxplot(ax=ax, column=column_stat,
                       boxprops=boxprops, medianprops=medianprops, sym='.')

xticklabels = [label[len(stat)+1:] for label in column_stat]
ax.set_xticklabels(xticklabels)

if stat == 'mae':
    ax.set_ylim([0,2])
    ax.set_title('abs. mean error [-]');
elif stat=='pbias':
    ax.set_ylim([-100,400])
    ax.set_title('%bias [-]');
elif stat=='corr':
    ax.set_ylim([-0.1,1])
    ax.set_title('correlation');

print('median values across gauges')
print(df.median())
    
plt.savefig(f"./Figures/NB4_Fig2_daily_{stat}_boxplot.png", dpi=150)

In [ ]:
# scatter pot
stat = 'pbias'

base = 'f09_f09_mg17_mosart'
target = ['f09_f09_rHDMA','f09_f09_rHDMAlk_mg17_dfw']

fig, ax = plt.subplots(1, figsize=(6.0,5))

if stat=='mae':
    xmin = 0.0; xmax=2.0
elif stat=='corr':
    xmin = -0.2; xmax=1.0
elif stat=='pbias':
    xmin = -1.; xmax=3

metric_base = daily_metrics_common[base][stat]

for case in target:
    grid_name = cases[case]
    metric_value = daily_metrics_common[case][stat]
    plt.scatter(metric_base, metric_value, c=case_meta[grid_name]['color'], label=f'{case}', marker='o', s=10, alpha=0.5, edgecolors='none')

    mask = ~np.isnan(metric_value) & ~np.isnan(metric_base)
    lr =  stats.linregress(metric_base[mask] , metric_value[mask])
    x=np.linspace(xmin,xmax, num=10, endpoint=True)
    ax.plot(x, lr.intercept + lr.slope*x, '--', lw=2, c=case_meta[grid_name]['color'])
    ax.axline((1, 1), slope=1, color="k", lw=1, ls=':')
    
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.xlabel(f'{stat} {base}')
plt.ylabel(f'{stat}')
plt.legend(fontsize='small');
if figureSave:
    plt.savefig(f"Figures/NB3_Fig3_daily_{stat}_scatter.png", dpi=150)

In [ ]:
#cdf
stat = 'corr'

gage_sets = ['common']   # common or/and indiv_network

fig, ax = plt.subplots(1, figsize=(6.0,5))

if stat=='mae':
    xmin = 0; xmax=2
elif stat=='corr':
    xmin = -0.5; xmax=1.0
elif stat=='pbias':
    xmin = -1.; xmax=5

for case, grid_name in cases.items():
    for gage_set in gage_sets:
        if gage_set=='indiv_network':
            ls = '--'
            metric_value = daily_metrics[case][stat]
        if gage_set=='common':
            ls = '-'
            metric_value = daily_metrics_common[case][stat]
            
        metric_value = metric_value[~np.isnan(metric_value)]
        xdata_sort = np.sort(metric_value)
        prob_metric=np.arange(1,float(len(xdata_sort)+1))/(1+len(xdata_sort))
    
        plt.plot(xdata_sort, prob_metric, c=case_meta[grid_name]['color'], ls=ls, label=f'{case}_{gage_set}', linewidth=1.25)
plt.xlim(xmin, xmax)
plt.ylim(0, 1)
plt.xlabel(f'{stat}')
plt.ylabel(f'F(X)')
plt.legend(fontsize='small');
if figureSave:
    plt.savefig(f"NB3_Fig4_daily_{stat}_cdf.png", dpi=150)